# ClearWater-Modules Demo 3: Coupling Water Quality Reactions to Transport with ClearWater-Riverine

**Objective**: Demonstrate a more complex scenario of coupled transport and reaction models in Sumwere Creek, using the [ClearWater-modules](https://github.com/EcohydrologyTeam/ClearWater-modules) to simulate heat exchange with the atmosphere.

This notebook builds on the introduction to using [ClearWater-riverine](https://github.com/EcohydrologyTeam/ClearWater-riverine) provided in the demo notebook in that reposittory.

## Background 
This notebook couples Clearwater-riverine (transport) with Clearwater-modules (reactions) - specifically, the Temperature Simulation Model (TSM). 

The Temperature Simulation Module (TSM) is an essential component of ClearWater (Corps Library for Environmental Analysis and Restoration of Watersheds). TSM plays a crucial role in simulating and predicting water temperature within aquatic ecosystems. TSM utilizes a comprehensive energy balance approach to account for various factors contributing to heat inputs and outputs in the water environment. It considers both external forcing functions and heat exchanges occurring at the water surface and the sediment-water interface. The primary contributors to heat exchange at the water surface include shortwave solar radiation, longwave atmospheric radiation, heat conduction from the atmosphere to the water, and direct heat inputs. Conversely, the primary factors that remove heat from the system are longwave radiation emitted by the water, evaporation, and heat conduction from the water to the atmosphere. 

The core principle behind TSM is the application of the laws of conservation of energy to compute water temperature. This means that the change in heat content of the water is directly related to changes in temperature, which, in turn, are influenced by various heat flux components. The specific heat of water is employed to establish this relationship. Each term of the heat flux equation can be calculated based on the input provided by the user, allowing for flexibility in modeling different environmental conditions

## Example Case Study

This example shows how to run Clearwater Riverine coupled with Clearwater Modules in a fictional location, "Sumwere Creek" (shown below). The flow field for Sumwere Creek comes from a HEC-RAS 2D model, which has a domain of 2x2 km and a base mesh cell size of 100x100 meters. 

![image.png](../docs/imgs/SumwereCreek_coarse.png)

The upstream boundary for Sumwere Creek is at the top left of the model domain, flowing into the domain at a constant 3 cms. At the first bend in the creek, there is an additional boundary representing a spring-fed tributary to the creek (1 cms). Further downstream, there is a meander in the stream forming a slow-flowing oxbow lake. There is another boundary flowing into that oxbow lake, representing a powerplant discharge (0.5 cms). 

The downstream boundary is a constant stage set at 20.75. The upstream inflows have a water temperature of 15 degrees C; the spring-fed creek has constant inflows of 10 C, and the powerplant is steady at 20 C with periodic higher temperature (25 C) discharges in a downstream meander.  

We simulate this scenario over the course of two full days, using meteorological parameters from Arizona (extreme temperature swings between night and day) to help show off the impacts of TSM.

## Data Availability
All data required to run this notebook is available in this repository at `examples\data\sumwere_creek_coarse`

# Setup

Carefully follow our **[Installation Instructions](https://github.com/EcohydrologyTeam/ClearWater-modules?tab=readme-ov-file#installation)**.

## Python Imports


In [1]:
from pathlib import Path
from typing import Optional
import yaml

import pandas as pd
import xarray as xr

import hvplot.xarray
import hvplot.pandas
import holoviews as hv
from bokeh.models import HoverTool

import clearwater_modules_v2 as cwm
from clearwater_modules_v2.config import init_from_file
from clearwater_riverine.plotting import RiverinePlotter

## Configure Models: Clearwater-Modules & Clearwater-Riverine

All data required to run this notebook is available in this repository at `examples\data\sumwere_creek_coarse`

This example sets up the model using a config file. The config files are structured to simulate "water_temperature" and "water_temperature_transport". The output variable "water_temperature" are the model results from the linked Riverine/TSM simulation and represents the transport from the Riverine model in addition to the comprehensive energy balance from the TSM model. While, the output variable "water_temperature_transport" represents only the transport from the Riverine model.

### Set paths 

In [2]:
model_name = 'sumwere_creek_coarse'

In [3]:
# Simulation directory path for inputs and outputs
sim_path = Path.cwd() / 'data' / model_name

#### Set local filepath to the configuration yaml file ####
config_filepath = sim_path / 'modules.yml'
config_filepath.exists()

True

In [4]:
# Display and confirm config settings
# Edit config file directly in a code editor
with open(config_filepath) as f:
    config = yaml.safe_load(f)
config

{'model': {'start_datetime': '2022-05-13 08:00:00',
  'end_datetime': '2022-05-14 20:00:00',
  'time_step': '30s',
  'chunk_time_step': '36hr',
  'simulation_directory': 'data/sumwere_creek_coarse/',
  'output_variables': ['water_temperature',
   'water_temperature_transport',
   'tracer']},
 'processes': [{'riverine': {'configuration_path': 'riverine.yml',
    'time_step': '30s'}},
  {'temperature': {'wind_a': 0.3,
    'wind_b': 1.5,
    'wind_c': 3.0,
    'time_step': '30s',
    'use_sediment_temperature': False}}],
 'data_sources': {'air_temp': {'provider': 'csv',
   'data': {'file_path': 'cwr_boundary_conditions_TairC_p28.csv',
    'time_field': 'Datetime'}},
  'solar': {'provider': 'csv',
   'data': {'file_path': 'cwr_boundary_conditions_q_Solar_p28.csv',
    'time_field': 'Datetime'}},
  'cloudiness': {'provider': 'float', 'data': {'value': 0.1}},
  'wind_speed': {'provider': 'float', 'data': {'value': 3.0}},
  'atmospheric_pressure': {'provider': 'float', 'data': {'value': 1013.

### Initialize from File

Clearwater-Modules is instantiated with a config file which contains information about the `model`, `processes`, and `data_sources`.

NOTE that riverine transport is coupled via a process, and has it's own config file.

In [5]:
#### Initialize a version 2 model of the linked Riverine and TSM models ####
model = init_from_file(config_filepath)

### Run the Coupled Models

In [6]:
%%time
#### Simulate a version 2 model of the linked Riverine and TSM models ####
model.run()

CPU times: user 1min 35s, sys: 2.16 s, total: 1min 37s
Wall time: 1min 36s


# Analyze Inputs & Outputs from Zarr File

In [7]:
# This is the prefered approach to explore model outputs
input_ds = xr.open_zarr(sim_path / "model_inputs.zarr", consolidated=False)
input_ds

<xarray.Dataset> Size: 415kB
Dimensions:                     (time: 4321)
Coordinates:
  * time                        (time) datetime64[ns] 35kB 2022-05-13T08:00:0...
Data variables:
    water_temperature           (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    sediment_thickness          (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    wetted_surface_area         (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    solar_radiation             (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    air_temperature             (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    cloudiness                  (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    volume                      (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    sediment_temperature        (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    wind_speed                  (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    atmospheric_vapor_pressure  (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>
    atmospheric_pressure        (time) float64 35kB dask.array<chunksize=(4321,), meta=np.ndarray>

In [8]:
# This is the prefered approach to explore model outputs
output_ds = xr.open_zarr(sim_path / "model_outputs.zarr", consolidated=False)
output_ds

<xarray.Dataset> Size: 46MB
Dimensions:                      (time: 4321, nface: 444)
Coordinates:
  * time                         (time) datetime64[ns] 35kB 2022-05-13T08:00:...
  * nface                        (nface) int64 4kB 0 1 2 3 4 ... 440 441 442 443
Data variables:
    water_temperature            (time, nface) float64 15MB dask.array<chunksize=(4320, 1), meta=np.ndarray>
    tracer                       (time, nface) float64 15MB dask.array<chunksize=(4320, 1), meta=np.ndarray>
    water_temperature_transport  (time, nface) float64 15MB dask.array<chunksize=(4320, 1), meta=np.ndarray>

# Plot the Coupled Models Results

## Map

In [9]:
#Initialize a Riverine dynamic plotting tool
plotter = RiverinePlotter(registry=model._Model__registry, crs='EPSG:26916')

In [10]:
#Plot the water temperature results from the linked Riverine and TSM model simulation
plotter.dynamic_plot(constituent_name = 'water_temperature')

BokehModel(combine_events=True, render_bundle={'docs_json': {'fd4b2d9f-b9b4-4123-85a0-bd5a55d89e39': {'version…

In [11]:
#Plot the water temperature results from just the Riverine model simulation
plotter.dynamic_plot(constituent_name = 'water_temperature_transport')

BokehModel(combine_events=True, render_bundle={'docs_json': {'8845e348-94a8-45ed-8ad3-a809e42ed6ef': {'version…

## Timeseries

In [12]:
# defaults for line plot
cells = {
    217: 'upstream reach, midway', 
    285: 'upstream reach just before spring',
    226: 'confluence of creek and spring',
    151: 'mixing zone power plant',  
    180: 'further from power plant',
    317: 'confluence below power plant',
}

In [13]:
def model_compare_vars_lines_plots(
    output_dataset: xr.Dataset, # model outputs
    variable_name_1: str, 
    variable_name_2: str,
    cells_dict: dict[int, str], #with key as cell intergers and value as cell location used for plot titles
):
    '''Holoviews line plot overlay for a given variable at selected grid cells.'''
    
    layout_curve_plots_list = []
    
    for cell, title_name in cells_dict.items():
        overlay_curve_plots_list = []
        ds_var1 = output_dataset[variable_name_1].isel(nface=cell)
        ds_var2 = output_dataset[variable_name_2].isel(nface=cell)
        curve_plot_1 = hv.Curve(ds_var1, label=f'cell {cell}; var1').opts(tools=['hover'])
        curve_plot_2 = hv.Curve(ds_var2, label=f'cell {cell}; var2').opts(tools=['hover'])
        overlay_curve_plots_list.append(curve_plot_1)
        overlay_curve_plots_list.append(curve_plot_2)        
        overlay_plot_i = hv.Overlay(overlay_curve_plots_list).opts(width=600, height=300, legend_position='right', title=title_name)
        layout_curve_plots_list.append(overlay_plot_i)

    return hv.Layout(layout_curve_plots_list).cols(1)

In [14]:
model_compare_vars_lines_plots(output_ds, variable_name_1 = 'water_temperature', variable_name_2 = 'water_temperature_transport', cells_dict=cells)

:Layout
   .Overlay.I   :Overlay
      .Curve.Cell_217_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_217_semicolon_var2 :Curve   [time]   (water_temperature_transport)
   .Overlay.II  :Overlay
      .Curve.Cell_285_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_285_semicolon_var2 :Curve   [time]   (water_temperature_transport)
   .Overlay.III :Overlay
      .Curve.Cell_226_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_226_semicolon_var2 :Curve   [time]   (water_temperature_transport)
   .Overlay.IV  :Overlay
      .Curve.Cell_151_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_151_semicolon_var2 :Curve   [time]   (water_temperature_transport)
   .Overlay.V   :Overlay
      .Curve.Cell_180_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_180_semicolon_var2 :Curve   [time]   (water_temperature_transport)
   .Overlay.VI  :Overlay
      .Curve.Cell_317_semicolon_var1 :Curve   [time]   (water_temperature)
      .Curve.Cell_317_semicolon_var2 :Curve   [time]   (water_temperature_transport)

# END